# Gun 1 - Config Tabanli Model Secimi Icin Factory Tasarimi (DOC-26)

`config/settings.yaml` uzerindeki `llm_settings.active_mode` (cloud/local) ve ilgili `provider` (anthropic/openai/huggingface) degerlerine gore dogru istemciyi kuran `src/llm_factory.py` modulunu test ediyoruz:

1. **Arayuz ve saglayici cozumleme** - `get_llm_client()`'in config'e gore dogru sinifi (`AnthropicClient`/`OpenAIClient`/`LocalHFClient`) urettigini ve hepsinin ortak `LLMClient` arayuzune uydugunu dogruluyoruz.
2. **Gecersiz config davranisi** - taninmayan bir `provider` veya eksik bir `model_name` icin acik `ValueError` firlatildigini dogruluyoruz.
3. **Cloud ucu gercek API ile uctan uca** - `active_mode: cloud` ile Anthropic'e gercekten baglanip yanit aliyoruz.
4. **Local ucun Gun 1 kapsami** - `active_mode: local` secildiginde Factory'nin dogru sinifa (`LocalHFClient`) yonlendirdigini, ama gercek inference'in henuz baglanmadigini (bilinctli `NotImplementedError`, bkz. DOC-27) dogruluyoruz.

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from llm_factory import (
    LLMClient,
    AnthropicClient,
    OpenAIClient,
    LocalHFClient,
    load_llm_config,
    get_llm_client,
)

print("llm_factory modulu yuklendi.")

llm_factory modulu yuklendi.


## 1. Config'ten saglayici cozumleme

In [2]:
config = load_llm_config()
print("settings.yaml -> llm_settings:", config)

client = get_llm_client(config)
print(f"active_mode='{config['active_mode']}' -> {type(client).__name__}")
assert isinstance(client, LLMClient)
assert isinstance(client, AnthropicClient)
print("OK - cloud/anthropic config'i dogru sinifi uretti ve LLMClient arayuzune uyuyor.")

settings.yaml -> llm_settings: {'active_mode': 'cloud', 'cloud_model': {'provider': 'anthropic', 'model_name': 'claude-sonnet-5'}, 'local_model': {'provider': 'huggingface', 'model_name': 'meta-llama/Meta-Llama-3-8B-Instruct'}}


active_mode='cloud' -> AnthropicClient
OK - cloud/anthropic config'i dogru sinifi uretti ve LLMClient arayuzune uyuyor.


In [3]:
import os

# openai istemcisi kurulurken (gercek cagri yapilmadan) API anahtari kontrolu
# yapiyor; burada sadece Factory dispatch mantigini (dogru sinifin secildigini)
# test ediyoruz, gercek API cagrisi yapmiyoruz -- bu yuzden gecici bir sahte
# anahtar veriyoruz.
os.environ.setdefault("OPENAI_API_KEY", "sk-test-dummy-key-for-factory-dispatch-test")

openai_config = {
    "active_mode": "cloud",
    "cloud_model": {"provider": "openai", "model_name": "gpt_4"},
}
openai_client = get_llm_client(openai_config)
assert isinstance(openai_client, OpenAIClient)
print(f"OK - provider='openai' (orn. gpt_4) -> {type(openai_client).__name__}")

local_config = {
    "active_mode": "local",
    "local_model": {"provider": "huggingface", "model_name": "local_llama"},
}
local_client = get_llm_client(local_config)
assert isinstance(local_client, LocalHFClient)
print(f"OK - active_mode='local' (orn. local_llama) -> {type(local_client).__name__}")

print("Ucu de LLMClient arayuzune uyuyor:", all(isinstance(c, LLMClient) for c in [client, openai_client, local_client]))


OK - provider='openai' (orn. gpt_4) -> OpenAIClient
OK - active_mode='local' (orn. local_llama) -> LocalHFClient
Ucu de LLMClient arayuzune uyuyor: True


## 2. Gecersiz config davranisi

In [4]:
try:
    get_llm_client({"active_mode": "cloud", "cloud_model": {"provider": "bilinmeyen_saglayici", "model_name": "x"}})
    print("FAIL - taninmayan provider icin ValueError beklenirdi")
except ValueError as e:
    print(f"OK - taninmayan provider icin ValueError firlatildi: {e}")

try:
    get_llm_client({"active_mode": "cloud", "cloud_model": {"provider": "anthropic"}})
    print("FAIL - eksik model_name icin ValueError beklenirdi")
except ValueError as e:
    print(f"OK - eksik model_name icin ValueError firlatildi: {e}")

try:
    get_llm_client({"active_mode": "local"})
    print("FAIL - tanimsiz local_model icin ValueError beklenirdi")
except ValueError as e:
    print(f"OK - tanimsiz local_model icin ValueError firlatildi: {e}")

OK - taninmayan provider icin ValueError firlatildi: Bilinmeyen LLM saglayicisi: 'bilinmeyen_saglayici'. Desteklenenler: ['anthropic', 'openai', 'huggingface']
OK - eksik model_name icin ValueError firlatildi: config/settings.yaml icinde 'cloud_model.model_name' tanimli degil.
OK - tanimsiz local_model icin ValueError firlatildi: config/settings.yaml icinde active_mode='local' icin 'local_model' tanimli degil.


## 3. Cloud ucu: gercek API ile uctan uca test

In [5]:
cloud_client = get_llm_client()  # settings.yaml -> active_mode: cloud (anthropic)

try:
    answer = cloud_client.generate(
        system_prompt="Kisa ve net cevap ver.",
        user_message="Turkiye'nin baskenti neresidir? Tek kelimeyle cevapla.",
    )
    print(f"OK - Factory'nin urettigi istemci ({type(cloud_client).__name__}) gercek API'den yanit aldi:")
    print(answer)
except Exception as e:
    print(
        "Gercek API cagrisi basarisiz oldu (muhtemelen ANTHROPIC_API_KEY "
        f".env dosyasinda tanimli degil): {e}"
    )

OK - Factory'nin urettigi istemci (AnthropicClient) gercek API'den yanit aldi:
Ankara


## 4. Local ucun Gun 1 kapsami (guncelleme notu: DOC-27'de gercek baglanti eklendi)

`active_mode: local` secildiginde Factory dogru sinifa (`LocalHFClient`) yonlendiriyor. Gun 1'de (bu notebook ilk yazildiginda) `generate()` bilincli olarak `NotImplementedError` firlatiyordu; DOC-27 (Gun 2) kapsaminda gercek transformers baglantisi eklendi (bkz. `notebooks/11_llm_factory_cloud_local_integration_test.ipynb`).

Asagida config'teki varsayilan local model (`meta-llama/Meta-Llama-3-8B-Instruct`) ile deniyoruz: bu model **gated** oldugu ve bu ortamda `HF_TOKEN`/erisim onayi olmadigi icin gercekci bir hata (`OSError` - erisim reddedildi) bekliyoruz; `NotImplementedError` degil, artik gercekten baglanmaya calisip HuggingFace'in erisim kontrolune takiliyor. Herkese acik bir modelle gercek basarili calistirma notebook 11'de gosteriliyor.

In [6]:
local_config = dict(config)
local_config["active_mode"] = "local"
local_client = get_llm_client(local_config)
print(f"active_mode='local' -> {type(local_client).__name__} (model_name='{local_client.model_name}')")

try:
    local_client.generate("sys", "usr")
    print("FAIL - gated model icin erisim hatasi beklenirdi")
except NotImplementedError as e:
    print(f"NotImplementedError (Gun 1 davranisi, artik guncel degil): {e}")
except OSError as e:
    print("OK - gated model icin gercekci bir erisim hatasi alindi (kod calisiyor, sadece")
    print("      bu ortamda bu modele erisim/HF_TOKEN yok). Herkese acik bir modelle basarili")
    print("      calistirma icin bkz. notebooks/11_llm_factory_cloud_local_integration_test.ipynb.")
    print(f"      Hata ozeti: {type(e).__name__}")

active_mode='local' -> LocalHFClient (model_name='meta-llama/Meta-Llama-3-8B-Instruct')


OK - gated model icin gercekci bir erisim hatasi alindi (kod calisiyor, sadece
      bu ortamda bu modele erisim/HF_TOKEN yok). Herkese acik bir modelle basarili
      calistirma icin bkz. notebooks/11_llm_factory_cloud_local_integration_test.ipynb.
      Hata ozeti: OSError


## Ozet

- `LLMClient` (ABC) ortak arayuzu tanimlandi: `generate(system_prompt, user_message, max_tokens) -> str`.
- `get_llm_client()` Factory'si `config/settings.yaml` -> `llm_settings.active_mode`'a gore dogru saglayiciyi (`AnthropicClient` / `OpenAIClient` / `LocalHFClient`) kuruyor; yeni bir saglayici eklemek icin sadece `_PROVIDER_REGISTRY` genisletilecek.
- Cloud ucu (Anthropic) gercek API ile uctan uca dogrulandi; `provider: openai` (orn. `gpt_4`) icin de dogru sinif urettigi gosterildi.
- Gecersiz config (bilinmeyen provider, eksik model_name, tanimsiz mod bloğu) acik `ValueError` ile erken hata veriyor.
- **Guncelleme (DOC-27):** local (huggingface) ucu artik gercekten calisiyor (bu notebook yazildiginda iskeletti). Gated modeller icin gercekci erisim hatasi aliniyor; herkese acik modellerle basarili calistirma ve loglama icin bkz. `notebooks/11_llm_factory_cloud_local_integration_test.ipynb` ve `notebooks/12_multi_model_e2e_chain_test.ipynb`.